# DMOSpeech 2: Follow-up Tests on the Colab T4
## Fresh held-out speakers, degraded prompts, and the seed-noise floor

**Thesis:** A Risk-Calibrated Adaptive Handover Method for Teacher-Guided Flow-Matching TTS Inference

The tests recommended by the 25 October audit (`research/audits/2026-10-25/FINDINGS.md` in the thesis repository). Synthesis and scoring use the same sampler, metrics, checkpoints, dataset and seeds as the 1,000-utterance T4 grid (`gpu_grid_all1000.csv`).

| Test | Question | Speakers | Cases |
|---|---|---|---|
| **H1** Fresh held-out grid | Does the policy calibrated on the T4 calibration split keep risk ≤ 5% on speakers nobody has looked at? | 166 Seed-TTS speakers in no split | 332 (clean + noisy15) × Steps 4/5/6/8 |
| **H2** Degraded prompts | Does drift catch failures the SNR gate misses? Gaussian 18/20/22/25 dB, DEMAND cafeteria babble 15/20 dB, reverberation (RT60 0.3/0.9 s), telephone band | Same 166 | 1,494 × Steps 4/5/6/8 |
| **H3** Seed-noise floor | How often does Step 8 "fail" against Step 8 with another seed? (Replaces the unsupported 2.5% figure.) | 200 test | 400 × 2 seeds |

**Protocol** frozen in `preregistration.json` (Step 7) before any test runs. After freezing, the runner refuses to start if the protocol, the split manifests, the committed T4 grid or the runner itself change. **Analysis** computes every number from the CSV files; missing data stops it instead of being filled in.

**Cost of the run:** roughly 5–7 hours on a T4 (an estimate, not measured). Everything is resumable: results are flushed to Google Drive after every case.

---
### Instructions
1. Set the runtime to **GPU** (`Runtime > Change runtime type > T4 GPU`).
2. **Runtime > Run all.** Step 8 runs in the background; if the browser disconnects, reconnect and run **Step 8b**.
3. After Step 7, download `preregistration.json` and `fresh_manifest.json` from `MyDrive/dmospeech2_results/followup_t4/` and commit them to `research/experiments/EXP-001-followup-t4/` in the thesis repository.
4. The last cell downloads `DMOSpeech2_Followup_Results.zip`; commit its contents to `research/experiments/EXP-001-followup-t4/results/`.

### Step 1: Verify GPU Allocation

In [ ]:
!nvidia-smi

### Step 2: Clone Repositories & Install Dependencies
`Shonget-DMOSpeech2` is checked out at `8a7868f`, the commit whose `core/` sampler produced the T4 grid (it also contains the `656d86f` fix that makes SIM honour the ground-truth sample rate). The thesis repository (branch `rhenelAudit`) supplies the runner, the split manifests and the T4 grid; if it is private, add a Colab secret named `GITHUB_TOKEN`.

If Colab shows **"Restart session"** after the installs, click it, then continue from **Step 3** (no need to re-run Step 2). Every later step re-enters the repository folder and re-reads its settings, so a restart is safe.

In [ ]:
import os, subprocess
DMO_COMMIT = '8a7868fc3345702a586173447f4869205b501f26'
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git fetch -q origin && git checkout -q {DMO_COMMIT}
!git log --oneline -1

token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
THESIS_BRANCH = 'rhenelAudit'   # the branch that holds the follow-up runner
if not os.path.exists('/content/Final-Thesis-Setup'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git",
                    '/content/Final-Thesis-Setup'], check=True)
!git -C /content/Final-Thesis-Setup fetch -q origin {THESIS_BRANCH} && git -C /content/Final-Thesis-Setup checkout -q {THESIS_BRANCH} && git -C /content/Final-Thesis-Setup pull -q origin {THESIS_BRANCH}
!git -C /content/Final-Thesis-Setup log --oneline -1

# Same pinned dependencies as the 1,000-utterance GPU retest notebook
!pip install -q "numpy==2.2.3" "pandas==2.2.3"
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub

import numpy as np, pandas as pd, numba
print(f"Verified: NumPy {np.__version__} | Pandas {pd.__version__} | Numba {numba.__version__} [OK]")

runner_path = '/content/Final-Thesis-Setup/code/colab/run_followup_tests.py'
if not os.path.exists(runner_path):
    raise FileNotFoundError(f"{runner_path} is not on branch {THESIS_BRANCH}. Push it to GitHub before continuing.")

### Step 3: Download DMOSpeech 2 Checkpoints
`model_85000.pt` (4.0 GB, teacher + student) and `model_1500.pt` (270 MB, duration predictor) into `ckpts/`, checked against the SHA-256 hashes recorded for the earlier runs.

In [ ]:
%cd /content/Shonget-DMOSpeech2
!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

import hashlib
for name, digest in {'model_85000.pt': 'e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c',
                     'model_1500.pt': '6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2'}.items():
    h = hashlib.sha256()
    with open(f'ckpts/{name}', 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    assert h.hexdigest() == digest, f"{name}: SHA-256 mismatch (delete it and re-run this step)"
    print(f"{name}: SHA-256 verified")

### Step 4: Download the Seed-TTS test-en Dataset
Prompt and ground-truth audio for all 666 speakers: the 500 in the splits and the 166 unused ones that form the fresh held-out set.

In [ ]:
import os
os.chdir('/content/Shonget-DMOSpeech2')

test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'

# Check if audio is missing or only a Git LFS pointer
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    print("Resolving Git LFS binaries for the Seed-TTS evaluation dataset...")
    if os.path.exists('data/seed-tts-eval/.git'):
        !cd data/seed-tts-eval && git lfs install && git lfs pull
    if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
        print("Cleaning leftover pointer files and downloading via huggingface_hub...")
        !rm -rf data/seed-tts-eval
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id="zhaochenyang20/seed-tts-eval", repo_type="dataset", local_dir="data/seed-tts-eval")
    print("Dataset setup complete!")
else:
    print("Seed-TTS audio binaries verified!")

!ls -lh data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav
!wc -l data/seed-tts-eval/en/meta.lst

### Step 5: Download DEMAND Cafeteria Babble
Real recorded noise for the `babble15` / `babble20` conditions: DEMAND (Thiemann, Ito and Vincent, 2013), environment PCAFETER, 16 kHz, channel 1, from Zenodo record 1227121. The runner records its SHA-256 with the results.

In [ ]:
import os
DEMAND = '/content/demand'
if not os.path.exists(f'{DEMAND}/PCAFETER/ch01.wav'):
    os.makedirs(DEMAND, exist_ok=True)
    !wget -q -c -O /content/PCAFETER_16k.zip "https://zenodo.org/records/1227121/files/PCAFETER_16k.zip?download=1"
    !unzip -q -o /content/PCAFETER_16k.zip -d {DEMAND}
if not os.path.exists(f'{DEMAND}/PCAFETER/ch01.wav'):
    !find {DEMAND} -maxdepth 3 | head -20
    raise FileNotFoundError("PCAFETER/ch01.wav not found: check the archive layout printed above")
!ls -lh {DEMAND}/PCAFETER/ch01.wav

### Step 6: Smoke Test (2 development speakers, a few minutes)
Checks models, dataset, every prompt condition and the seed path end to end. Uses development speakers only, so it never touches the fresh held-out speakers, and writes to a **separate** `smoke/` folder.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'followup_t4')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_followup_tests.py"'
DEMAND = '/content/demand'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

!python {RUNNER} smoke --out "{OUT}" --dmo_root {DMO} --demand_dir {DEMAND}

import pandas as pd
smoke = pd.read_csv(f'{OUT}/smoke/smoke_grid.csv')
print(f"Smoke test OK: {len(smoke)} rows, {smoke.shape[1]} columns; ground-truth sample rates: {sorted(smoke.gt_sr.unique())}")
display(smoke[['utt_id', 'condition', 'teacher_step', 'prompt_snr', 'd_4', 'wer', 'similarity', 'utmos', 'is_failure']])

### Step 7: Freeze the Protocol
Selects the 166 fresh speakers and writes `preregistration.json` and `fresh_manifest.json` to Drive. Runs once: if the protocol is already frozen, this cell only prints it. **Commit both files to the thesis repository** (`research/experiments/EXP-001-followup-t4/`).

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'followup_t4')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_followup_tests.py"'
DEMAND = '/content/demand'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

if not os.path.exists(f'{OUT}/preregistration.json'):
    !python {RUNNER} freeze --out "{OUT}" --dmo_root {DMO}
prereg = json.load(open(f'{OUT}/preregistration.json'))
print(f"Frozen {prereg['created_utc']} | sha256 {prereg['sha256']}")
print(f"Fresh speakers: {prereg['n_fresh_speakers']} of {prereg['n_source_speakers']} | thesis {prereg['thesis_commit'][:7]} | DMOSpeech2 {prereg['dmo_commit'][:7]}")
try:
    from google.colab import files
    files.download(f'{OUT}/preregistration.json')
    files.download(f'{OUT}/fresh_manifest.json')
except Exception:
    print(f"Download them from Drive: {OUT}")

### Step 8: Full Run (H1 → H3 → H2) + Analysis
Runs the fresh held-out grid, the seed-noise floor and the degraded-prompt conditions, then the analysis, in that order.

The job runs **in the background** and writes its full output to `MyDrive/dmospeech2_results/followup_run_log.txt`; this cell prints one progress line per minute. If only the browser disconnects, the job keeps running on the Colab machine; reconnect and run **Step 8b** to check progress.

**If the Colab machine itself is reset** (repositories, checkpoints, dataset and DEMAND deleted), the results so far are safe in Google Drive (`MyDrive/dmospeech2_results/followup_t4/`). Re-run Steps 1–5, skip Steps 6 and 7, then run this cell again: every test continues from the last saved case.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'followup_t4')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_followup_tests.py"'
DEMAND = '/content/demand'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

# Run in the background: all output goes to a log file in Drive, not to the browser.
# Thousands of per-clip warnings in the cell output can bloat the tab and drop the session.
import subprocess, time, re
if not os.path.exists(f'{OUT}/preregistration.json'):
    raise FileNotFoundError("Freeze the protocol first (Step 7).")
log_path = os.path.join(persist, 'followup_run_log.txt')
base = f'python {RUNNER} {{}} --out "{OUT}" --dmo_root {DMO}'
cmd = ' && '.join([base.format('fresh-grid'), base.format('seed-floor'),
                   base.format('degradations') + f' --demand_dir {DEMAND}',
                   f'python {RUNNER} analyze --out "{OUT}"'])
running = subprocess.run(['pgrep', '-f', 'run_followup_tests.py'], capture_output=True, text=True).stdout.split()
if running:
    # A job from before a browser disconnect is still going: monitor it, never start a second one
    pid = int(running[0])
    alive = lambda: os.path.exists(f'/proc/{pid}')
    print(f"A job is already running (pid {pid}); monitoring it instead of starting a new one.")
else:
    proc = subprocess.Popen(f'nohup bash -c \'{cmd}\' >> "{log_path}" 2>&1', shell=True)
    alive = lambda: proc.poll() is None
    print(f"Started (pid {proc.pid}). Full log: {log_path}")

# Light monitor: one progress line per minute until the job ends
last = ''
while alive():
    time.sleep(60)
    try:
        lines = open(log_path, encoding='utf-8', errors='ignore').read().splitlines()
        prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l or 'refusing' in l]
        if prog and prog[-1] != last:
            last = prog[-1]
            print(time.strftime('%H:%M'), last)
    except FileNotFoundError:
        pass
print(f"Finished{'' if running else f' with exit code {proc.returncode}'}. Last lines of the log:")
for line in open(log_path, encoding='utf-8', errors='ignore').read().splitlines()[-15:]:
    print(line)

### Step 8b: Check Progress After a Reconnect
Shows whether the background job is still running, the latest progress and how many cases each test has saved. Does **not** start a second job.

In [ ]:
import os, re, subprocess, csv
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'followup_t4')
log_path = os.path.join(persist, 'followup_run_log.txt')
running = subprocess.run(['pgrep', '-f', 'run_followup_tests.py'], capture_output=True, text=True).stdout.split()
print("Job running:", "yes (pid " + ", ".join(running) + ")" if running else "no")
if os.path.exists(log_path):
    lines = open(log_path, encoding='utf-8', errors='ignore').read().splitlines()
    prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
    print("Latest progress:", prog[-1] if prog else "(none yet)")
    for line in lines[-5:]:
        print(line)
else:
    print("No log yet (mount Drive with Step 6 or 8 first).")
for name, per_case, total in (('fresh_grid.csv', 4, 332), ('seed_floor.csv', 2, 400), ('degradations.csv', 4, 1494)):
    path = os.path.join(OUT, name)
    n = sum(1 for _ in open(path, encoding='utf-8')) - 1 if os.path.exists(path) else 0
    print(f"{name}: about {n // per_case} of {total} cases saved")

### Step 9: View the Report

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'followup_t4')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_followup_tests.py"'
DEMAND = '/content/demand'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

from IPython.display import Markdown, display
rep = os.path.join(OUT, 'FOLLOWUP_REPORT.md')
if not os.path.exists(rep) and os.path.exists(os.path.join(OUT, 'degradations.csv')):
    !python {RUNNER} analyze --out "{OUT}"
if os.path.exists(rep):
    display(Markdown(open(rep, encoding='utf-8').read()))
    s = json.load(open(os.path.join(OUT, 'analysis_results.json')))
    print("lambda_hat:", s['calibration']['lambda_hat'], "| partial:", s['partial'])
else:
    print(f"No report yet at {rep}. Run Step 8 first (or run `analyze --allow_partial` for an interim look).")

### Step 10: Zip All Outputs and Download
Packs the CSV files, preregistration, fresh manifest, analysis JSON, report, run information and the runner script into one ZIP, also saved to Drive. The failure audio in `listening/` stays on Drive for the listening check (audio is not committed to the repository).

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, json, torch
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
persist = '/content/drive/MyDrive/dmospeech2_results'
OUT = os.path.join(persist, 'followup_t4')
os.makedirs(OUT, exist_ok=True)
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'"{THESIS}/code/colab/run_followup_tests.py"'
DEMAND = '/content/demand'
os.chdir(DMO)
print(f"Results in Google Drive: {OUT}")

import zipfile, subprocess, datetime
zip_name = 'DMOSpeech2_Followup_Results.zip'
with open(os.path.join(OUT, 'RUN_INFO.txt'), 'w') as f:
    f.write(f"dmospeech2_commit: {subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()}\n")
    f.write(f"thesis_commit: {subprocess.run(['git', '-C', THESIS, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()}\n")
    f.write(f"zipped_at_utc: {datetime.datetime.utcnow().isoformat()}Z\n")
    f.write(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files_ in os.walk(OUT):
        dirs[:] = [d for d in dirs if d != 'listening']
        for fname in files_:
            if fname.startswith('tmp_') or fname.endswith('.wav'):
                continue
            full = os.path.join(root, fname)
            zf.write(full, os.path.relpath(full, OUT))
    zf.write(f'{THESIS}/code/colab/run_followup_tests.py', 'code/run_followup_tests.py')

print(f"Created {zip_name} ({os.path.getsize(zip_name) / 1e6:.2f} MB)")
with zipfile.ZipFile(zip_name) as zf:
    for n in zf.namelist():
        print("  ", n)

!cp "{zip_name}" "{persist}/"
print(f"Also saved to Google Drive: {persist}/{zip_name}")
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print(f"Download manually from the Files panel: /content/Shonget-DMOSpeech2/{zip_name}")